# SLM vs MRCD on 120/80 Split

This notebook:
1. Loads preprocessed data
2. Builds 120/80 split from first 200 samples by time
3. Fine-tunes SLM on train (120), evaluates on test (80)
4. Runs MRCD round-by-round in a single logic cell (no pipeline wrapper call)
5. Exports logs and compares SLM-only vs MRCD

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Replace "YOUR_HF_TOKEN" with your actual token
login(token=user_secrets.get_secret("HF_TOKEN"))
print("login")

In [ ]:
import os

GITHUB_REPO = "https://github.com/Chinh-de/Fake-news-detection.git"  # CẬP NHẬT TẠI ĐÂY
REPO_NAME = "Fake-news-detection"

if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO}
%cd {REPO_NAME}

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# Giữ nguyên ROOT_DIR theo yêu cầu, mọi truy vấn trong repo dùng REPO_DIR
ROOT_DIR = Path('..').resolve()
REPO_DIR = ROOT_DIR / 'Fake-news-detection'
if str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))

from src.utils import preprocess_text, log_prediction_to_csv, log_round_trace_to_csv
from src.slm.model import IntegratedSLM
from src.pipeline.evidence import prefetch_query_context, build_evidence_bundle, assess_with_llm
from src.pipeline.selection import split_clean_noisy, finalize_remaining_noisy_with_slm
from src.retrieval.demo_retrieval import load_news_corpus
from src.llm.handler import get_llm
from src.config import (
    NUM_LOOP, CONFIDENCE_THRESHOLD, TOP_K_DEMOS, FACT_TOP_K,
    ENABLE_SLM_FINETUNE, SLM_FINETUNE_EPOCHS, SLM_FINETUNE_BATCH_SIZE,
    SLM_FINETUNE_LR, SLM_FINETUNE_WEIGHT_DECAY, SLM_FINETUNE_MIN_SAMPLES,
)

# Log lưu ngoài repo
LOG_DIR = ROOT_DIR / 'log'
LOG_DIR.mkdir(parents=True, exist_ok=True)

SLM_PRED_CSV = LOG_DIR / 'slm_test_predictions.csv'
MRCD_PRED_CSV = LOG_DIR / 'mrcd_final_predictions.csv'
MRCD_ROUND_METRICS_CSV = LOG_DIR / 'mrcd_round_metrics.csv'
COMPARISON_CSV = LOG_DIR / 'slm_vs_mrcd_metrics.csv'
TRACE_CSV_PATH = LOG_DIR / 'mrcd_trace.csv'
RESULTS_CSV_PATH = LOG_DIR / 'mrcd_results.csv'

for p in [SLM_PRED_CSV, MRCD_PRED_CSV, MRCD_ROUND_METRICS_CSV, COMPARISON_CSV, TRACE_CSV_PATH, RESULTS_CSV_PATH]:
    if p.exists():
        p.unlink()

print(f'Root: {ROOT_DIR}')
print(f'Repo dir: {REPO_DIR}')
print(f'Log dir (outside repo): {LOG_DIR}')

### 1. Chuẩn bị dữ liệu và chia tập 120/80
Phần này nạp dữ liệu đã tiền xử lý, sắp theo thời gian và chia train/test theo mốc 120/80.

In [ ]:
DATA_PATH = REPO_DIR / 'dataset' / 'twitter15_16_preprocessed.csv'

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['text'] = df['text'].astype(str).apply(preprocess_text)
df = df.sort_values('timestamp').reset_index(drop=True)

df_200 = df.head(200).copy()
df_200['label_bin'] = df_200['label'].astype(str).str.lower().map(lambda x: 0 if x in ['true', 'non-rumor'] else 1)
df_200['split'] = ['train'] * 120 + ['test'] * 80

train_df = df_200[df_200['split'] == 'train'].copy()
test_df = df_200[df_200['split'] == 'test'].copy()

print(f'Total used: {len(df_200)} | Train: {len(train_df)} | Test: {len(test_df)}')
print('Train labels:', train_df['label_bin'].value_counts().to_dict())
print('Test labels:', test_df['label_bin'].value_counts().to_dict())

### 2. Trực quan phân phối theo thời gian
Vẽ timeline 200 mẫu đầu và biểu đồ phân phối nhãn cho train/test.

In [ ]:
# Time split and distribution
plt.figure(figsize=(14, 4))
for split, color in [('train', 'tab:blue'), ('test', 'tab:orange')]:
    subset = df_200[df_200['split'] == split]
    plt.scatter(subset['timestamp'], subset.index + 1, label=f'{split} (n={len(subset)})', alpha=0.75, color=color)

cut_time = df_200.iloc[119]['timestamp']
plt.axvline(cut_time, color='red', linestyle='--', label='cut at 120')
plt.title('First 200 samples over time with 120/80 split')
plt.xlabel('Timestamp')
plt.ylabel('Sample index (1..200)')
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

dist = df_200.groupby(['split', 'label_bin']).size().reset_index(name='count')
dist_pivot = dist.pivot(index='split', columns='label_bin', values='count').fillna(0)
dist_pivot.columns = ['label_0_real', 'label_1_fake']
dist_pivot.plot(kind='bar', figsize=(8, 5))
plt.title('Label distribution in train/test')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3. Fine-tune SLM và đánh giá trên tập test
Huấn luyện SLM trên train, dự đoán test và lưu kết quả SLM-only.

In [ ]:
# Fine-tune SLM on train, evaluate on test
train_texts = train_df['text'].tolist()
train_labels = train_df['label_bin'].astype(int).tolist()
test_texts = test_df['text'].tolist()
test_labels = test_df['label_bin'].astype(int).tolist()

SLM_SAVE_DIR = REPO_DIR / 'dataset' / 'slm_120_80'
SLM_SAVE_DIR.mkdir(parents=True, exist_ok=True)

slm = IntegratedSLM(model_path='roberta-base', backend='hf')
ft_stats = slm.fnetune(
    train_texts=train_texts,
    train_labels=train_labels,
    model_init='roberta-base',
    epochs=4,
    batch_size=32,
    lr=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    save_path=str(SLM_SAVE_DIR),
)

print('Fine-tune stats:', ft_stats)

plt.figure(figsize=(8, 4))
plt.plot(ft_stats['train_loss_history'], marker='o', color='tab:red')
plt.title('SLM training loss per epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

slm_results = slm.inference_batch(test_texts, batch_size=32)
slm_pred = [r[0] for r in slm_results]
slm_conf = [r[1] for r in slm_results]

slm_acc = accuracy_score(test_labels, slm_pred)
slm_prec, slm_rec, slm_f1, _ = precision_recall_fscore_support(test_labels, slm_pred, average='binary', zero_division=0)
print(f'SLM-only Test Accuracy: {slm_acc:.4f}')
print(f'SLM-only Precision: {slm_prec:.4f} | Recall: {slm_rec:.4f} | F1: {slm_f1:.4f}')
print(classification_report(test_labels, slm_pred, target_names=['real(0)', 'fake(1)'], digits=4))

cm = confusion_matrix(test_labels, slm_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['real', 'fake'], yticklabels=['real', 'fake'])
plt.title('SLM-only confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

pd.DataFrame({
    'event_id': test_df['id'].astype(str).tolist(),
    'timestamp': test_df['timestamp'].astype(str).tolist(),
    'text': test_texts,
    'y_true': test_labels,
    'y_pred_slm': slm_pred,
    'conf_slm': slm_conf,
}).to_csv(SLM_PRED_CSV, index=False, encoding='utf-8')
print(f'Saved: {SLM_PRED_CSV}')

### 4. MRCD nhiều vòng (Round-based)
Bên dưới tách thành từng bước: Round 1, Round 2..N, rồi chốt kết quả cuối.

In [ ]:
# MRCD - Phần A: Khởi tạo + Bootstrap + Round 1
events = test_texts
ground_truth = test_labels

slm_mrcd = IntegratedSLM(model_path=str(SLM_SAVE_DIR), backend='hf')
llm = get_llm()
static_corpus = load_news_corpus()

# Ổn định hơn trên Kaggle: chỉ dùng Wikipedia summary
mode = 'wiki_only'
wiki_fetch_full = False

event_states = [
    {
        'event_id': i,
        'text': preprocess_text(t),
        'round': 0,
        'status': 'unprocessed',
        'label': None,
        'label_llm': None,
        'label_slm': None,
        'conf_slm': None,
        'llm_raw': None,
        'llm_label_matched': None,
        'retrieval_source': None,
        'knowledge': None,
        'query_context': None,
        'ground_truth': ground_truth[i],
        'prompt': None,
    }
    for i, t in enumerate(events)
]

d_clean = []
d_noisy = []
round_history = []
finetune_history = []
round_eval_rows = []
knowledge_cache_local = {}

# Bootstrap context
unique_texts = list(dict.fromkeys([s['text'] for s in event_states]))
context_map = {}
for text in unique_texts:
    context_map[text] = prefetch_query_context(
        text=text,
        demo_k=TOP_K_DEMOS,
        fact_top_k=FACT_TOP_K,
        reuse_knowledge_cache=True,
        knowledge_cache_local=knowledge_cache_local,
        knowledge_mode=mode,
        wiki_fetch_full=wiki_fetch_full,
    )

for state in event_states:
    qctx = context_map.get(state['text'], {
        'knowledge_text': 'No info.',
        'bing_seed_news': [],
    })
    state['query_context'] = qctx
    state['knowledge'] = qctx.get('knowledge_text', 'No info.')

# Round 1
round_id = 1
for state in event_states:
    demos, knowledge_k, retrieval_source = build_evidence_bundle(
        text=state['text'],
        static_corpus=static_corpus,
        clean_pool=d_clean,
        round_id=round_id,
        query_context=state['query_context'],
        demo_k=TOP_K_DEMOS,
    )
    assess = assess_with_llm(text=state['text'], demos=demos, knowledge_k=knowledge_k, llm=llm)
    state.update({
        'round': round_id,
        'label': assess['y_llm'],
        'label_llm': assess['y_llm'],
        'llm_raw': assess['llm_raw'],
        'llm_label_matched': assess['llm_label_matched'],
        'retrieval_source': retrieval_source,
        'knowledge': knowledge_k,
        'prompt': assess['prompt'],
    })

slm_batch = slm_mrcd.inference_batch([s['text'] for s in event_states], batch_size=32)
for state, res in zip(event_states, slm_batch):
    pred, conf, _ = res
    state['label_slm'] = pred
    state['y_slm'] = pred
    state['conf_slm'] = conf

    log_round_trace_to_csv(
        round_id=round_id,
        event_id=state['event_id'],
        text=state['text'],
        y_slm=state['y_slm'],
        y_llm=state['label_llm'],
        ground_truth=state['ground_truth'],
        conf_slm=state['conf_slm'],
        prompt=state['prompt'],
        filepath=str(TRACE_CSV_PATH),
    )

    if split_clean_noisy(state, CONFIDENCE_THRESHOLD):
        state['status'] = 'clean'
        d_clean.append(state)
        log_prediction_to_csv(
            event_id=state['event_id'],
            text=state['text'],
            label=state['label'],
            conf=state['conf_slm'],
            round_id=round_id,
            status=state['status'],
            filepath=str(RESULTS_CSV_PATH),
        )
    else:
        state['status'] = 'noisy'
        d_noisy.append(state)

y_true_round = [s['ground_truth'] for s in event_states]
y_pred_round = [s['label'] for s in event_states]
acc_r = accuracy_score(y_true_round, y_pred_round)
pr_r, rc_r, f1_r, _ = precision_recall_fscore_support(y_true_round, y_pred_round, average='binary', zero_division=0)
round_eval_rows.append({'round': round_id, 'accuracy': acc_r, 'precision': pr_r, 'recall': rc_r, 'f1': f1_r, 'clean_count': len(d_clean), 'noisy_count': len(d_noisy)})

round_history.append({'round': round_id, 'clean_count': len(d_clean), 'noisy_count': len(d_noisy)})
print(f'Round 1 done | clean={len(d_clean)} noisy={len(d_noisy)}')

#### 4.1 MRCD - Round 2..N
Fine-tune trên d_clean (nếu đủ mẫu), sau đó tái đánh giá các mẫu noisy.

In [ ]:
# MRCD - Phần B: Rounds 2..N
round_id = 2
while d_noisy and round_id <= NUM_LOOP:
    if ENABLE_SLM_FINETUNE and len(d_clean) >= SLM_FINETUNE_MIN_SAMPLES:
        ft = slm_mrcd.finetune_on_clean(
            clean_samples=d_clean,
            epochs=SLM_FINETUNE_EPOCHS,
            batch_size=SLM_FINETUNE_BATCH_SIZE,
            lr=SLM_FINETUNE_LR,
            weight_decay=SLM_FINETUNE_WEIGHT_DECAY,
        )
    else:
        ft = {'trained': False, 'reason': 'disabled_or_insufficient'}
    finetune_history.append({'round': round_id, **ft})

    next_noisy = []
    promoted_clean = 0

    for state in d_noisy:
        demos, knowledge_k, retrieval_source = build_evidence_bundle(
            text=state['text'],
            static_corpus=static_corpus,
            clean_pool=d_clean,
            round_id=round_id,
            query_context=state['query_context'],
            demo_k=TOP_K_DEMOS,
        )
        assess = assess_with_llm(text=state['text'], demos=demos, knowledge_k=knowledge_k, llm=llm)
        state.update({
            'round': round_id,
            'label': assess['y_llm'],
            'label_llm': assess['y_llm'],
            'llm_raw': assess['llm_raw'],
            'llm_label_matched': assess['llm_label_matched'],
            'retrieval_source': retrieval_source,
            'knowledge': knowledge_k,
            'prompt': assess['prompt'],
        })

    slm_batch = slm_mrcd.inference_batch([s['text'] for s in d_noisy], batch_size=32)
    for state, res in zip(d_noisy, slm_batch):
        pred, conf, _ = res
        state['label_slm'] = pred
        state['y_slm'] = pred
        state['conf_slm'] = conf

        log_round_trace_to_csv(
            round_id=round_id,
            event_id=state['event_id'],
            text=state['text'],
            y_slm=state['y_slm'],
            y_llm=state['label_llm'],
            ground_truth=state['ground_truth'],
            conf_slm=state['conf_slm'],
            prompt=state['prompt'],
            filepath=str(TRACE_CSV_PATH),
        )

        if split_clean_noisy(state, CONFIDENCE_THRESHOLD):
            state['status'] = f'clean@round{round_id}'
            d_clean.append(state)
            promoted_clean += 1
            log_prediction_to_csv(
                event_id=state['event_id'],
                text=state['text'],
                label=state['label'],
                conf=state['conf_slm'],
                round_id=round_id,
                status=state['status'],
                filepath=str(RESULTS_CSV_PATH),
            )
        else:
            state['status'] = f'noisy@round{round_id}'
            next_noisy.append(state)

    d_noisy = next_noisy
    round_history.append({'round': round_id, 'promoted_to_clean': promoted_clean, 'clean_count': len(d_clean), 'noisy_count': len(d_noisy)})

    y_true_round = [s['ground_truth'] for s in event_states]
    y_pred_round = [s['label'] for s in event_states]
    acc_r = accuracy_score(y_true_round, y_pred_round)
    pr_r, rc_r, f1_r, _ = precision_recall_fscore_support(y_true_round, y_pred_round, average='binary', zero_division=0)
    round_eval_rows.append({'round': round_id, 'accuracy': acc_r, 'precision': pr_r, 'recall': rc_r, 'f1': f1_r, 'clean_count': len(d_clean), 'noisy_count': len(d_noisy)})

    print(f'Round {round_id} done | +clean={promoted_clean} remain_noisy={len(d_noisy)}')
    round_id += 1

#### 4.2 MRCD - Chốt kết quả cuối và lưu log
Final judgment cho mẫu noisy còn lại, tính metric cuối và xuất CSV.

In [ ]:
# MRCD - Phần C: Final judgment + export
finalized_noisy = []
if d_noisy:
    finalized_noisy = finalize_remaining_noisy_with_slm(d_noisy, slm_mrcd)
    for fs in finalized_noisy:
        fs['status'] = 'finalized_by_slm'
        log_round_trace_to_csv(
            round_id='final_judgment',
            event_id=fs['event_id'],
            text=fs['text'],
            y_slm=fs['label'],
            y_llm=None,
            ground_truth=fs['ground_truth'],
            conf_slm=fs['conf_slm'],
            prompt='N/A (Final SLM Judgment)',
            filepath=str(TRACE_CSV_PATH),
        )
        log_prediction_to_csv(
            event_id=fs['event_id'],
            text=fs['text'],
            label=fs['label'],
            conf=fs['conf_slm'],
            round_id=NUM_LOOP + 1,
            status=fs['status'],
            filepath=str(RESULTS_CSV_PATH),
        )

finalized_map = {x['event_id']: x for x in finalized_noisy}
mrcd_final_pred = []
mrcd_final_conf = []
mrcd_final_status = []
for s in sorted(event_states, key=lambda z: z['event_id']):
    if s['event_id'] in finalized_map:
        fs = finalized_map[s['event_id']]
        mrcd_final_pred.append(int(fs['label']))
        mrcd_final_conf.append(float(fs.get('conf_slm', 0.0)))
        mrcd_final_status.append('finalized_by_slm')
    else:
        mrcd_final_pred.append(int(s['label']))
        mrcd_final_conf.append(float(s.get('conf_slm', 0.0) or 0.0))
        mrcd_final_status.append(str(s.get('status', 'unknown')))

mrcd_acc = accuracy_score(ground_truth, mrcd_final_pred)
mrcd_prec, mrcd_rec, mrcd_f1, _ = precision_recall_fscore_support(ground_truth, mrcd_final_pred, average='binary', zero_division=0)

print(f'MRCD Final Accuracy: {mrcd_acc:.4f}')
print(f'MRCD Precision: {mrcd_prec:.4f} | Recall: {mrcd_rec:.4f} | F1: {mrcd_f1:.4f}')
print(classification_report(ground_truth, mrcd_final_pred, target_names=['real(0)', 'fake(1)'], digits=4))

round_eval_df = pd.DataFrame(round_eval_rows)
round_eval_df.to_csv(MRCD_ROUND_METRICS_CSV, index=False, encoding='utf-8')

pd.DataFrame({
    'event_id': test_df['id'].astype(str).tolist(),
    'timestamp': test_df['timestamp'].astype(str).tolist(),
    'text': test_texts,
    'y_true': ground_truth,
    'y_pred_mrcd': mrcd_final_pred,
    'conf_mrcd': mrcd_final_conf,
    'status_mrcd': mrcd_final_status,
}).to_csv(MRCD_PRED_CSV, index=False, encoding='utf-8')

print(f'Saved trace log: {TRACE_CSV_PATH}')
print(f'Saved result log: {RESULTS_CSV_PATH}')
print(f'Saved round metrics: {MRCD_ROUND_METRICS_CSV}')
print(f'Saved MRCD predictions: {MRCD_PRED_CSV}')

In [ ]:
# Round-by-round MRCD performance and final comparison
display(round_eval_df)

if not round_eval_df.empty:
    plt.figure(figsize=(8, 4))
    plt.plot(round_eval_df['round'], round_eval_df['accuracy'], marker='o', label='accuracy')
    plt.plot(round_eval_df['round'], round_eval_df['f1'], marker='o', label='f1')
    plt.title('MRCD performance by round')
    plt.xlabel('Round')
    plt.ylabel('Score')
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.show()

cm_mrcd = confusion_matrix(ground_truth, mrcd_final_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_mrcd, annot=True, fmt='d', cmap='Greens', xticklabels=['real', 'fake'], yticklabels=['real', 'fake'])
plt.title('MRCD final confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

comparison_df = pd.DataFrame([
    {'model': 'SLM_only', 'accuracy': slm_acc, 'precision': slm_prec, 'recall': slm_rec, 'f1': slm_f1},
    {'model': 'MRCD', 'accuracy': mrcd_acc, 'precision': mrcd_prec, 'recall': mrcd_rec, 'f1': mrcd_f1},
])
comparison_df.to_csv(COMPARISON_CSV, index=False, encoding='utf-8')
display(comparison_df)

ax = comparison_df.set_index('model')[['accuracy', 'f1']].plot(kind='bar', figsize=(7, 4))
ax.set_title('SLM-only vs MRCD')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.0)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'Saved comparison: {COMPARISON_CSV}')